# 04. Merging and Scheduling with Heaps

Heaps decide **what to do next** when many things compete: the next smallest from k lists, the room that frees up first, the most frequent task.

**Learning order:** MERGE K → MEETING ROOMS → CONNECT ROPES → TASK SCHEDULER → REORGANIZE STRING → INTERVIEW THINKING

## 1. Merge K Sorted Arrays

Heap of `(value, array index, element index)` with one element per array. Pop the smallest, then push the **next element of the same array**. O(N log k) for N total elements.

In [1]:
import heapq
from collections import Counter

def merge_k(arrays):
    heap = [(arr[0], i, 0) for i, arr in enumerate(arrays) if arr]
    heapq.heapify(heap)
    res = []
    while heap:
        val, i, j = heapq.heappop(heap)
        res.append(val)
        if j + 1 < len(arrays[i]):
            heapq.heappush(heap, (arrays[i][j + 1], i, j + 1))
    return res

print(merge_k([[1, 4, 5], [1, 3, 4], [2, 6]]))   # [1, 1, 2, 3, 4, 4, 5, 6]

[1, 1, 2, 3, 4, 4, 5, 6]


## 2. Meeting Rooms II (minimum rooms)

Sort meetings by start time. Keep a **min-heap of end times** (rooms in use). For each meeting:
- If the earliest ending room is free (`heap[0] <= start`), **reuse** it (replace its end time).
- Otherwise **open a new room**.

The heap size at the end = rooms needed. (A difference array / sweep line also works.)

In [2]:
def min_meeting_rooms(intervals):
    heap = []
    for start, end in sorted(intervals):
        if heap and heap[0] <= start:
            heapq.heapreplace(heap, end)         # reuse the free room
        else:
            heapq.heappush(heap, end)
    return len(heap)

print(min_meeting_rooms([[0, 30], [5, 10], [15, 20]]), min_meeting_rooms([[7, 10], [2, 4]]))   # 2 1

2 1


## 3. Minimum Cost to Connect Ropes (Huffman idea)

Joining two ropes costs their sum. To minimize the total, always join the **two shortest** ropes (long ropes should be added as few times as possible).

In [3]:
def connect_ropes(ropes):
    heapq.heapify(ropes)
    total = 0
    while len(ropes) > 1:
        a, b = heapq.heappop(ropes), heapq.heappop(ropes)
        total += a + b
        heapq.heappush(ropes, a + b)
    return total

print(connect_ropes([4, 3, 2, 6]))   # 29

29


## 4. Task Scheduler

Tasks with a **cooldown** `n` between the same task. Minimum time? The most frequent task decides the frame: `(max_freq - 1)` full cycles of length `n + 1`, plus the tasks that tie for the max frequency in the last row. The answer can never be smaller than the number of tasks.

`time = max(len(tasks), (max_freq - 1) * (n + 1) + count_of_max)`

(A heap + cooldown queue simulation gives the same answer and is the general method when you also need the order.)

In [4]:
def least_interval(tasks, n):
    counts = Counter(tasks)
    max_freq = max(counts.values())
    count_max = sum(1 for c in counts.values() if c == max_freq)
    return max(len(tasks), (max_freq - 1) * (n + 1) + count_max)

print(least_interval(["A", "A", "A", "B", "B", "B"], 2))        # 8
print(least_interval(["A", "C", "A", "B", "D", "B"], 1))        # 6
print(least_interval(["A", "A", "A", "B", "B", "B"], 0))        # 6

8
6
6


## 5. Reorganize String (no two adjacent equal)

Always place the **most frequent remaining** character that is not the one just placed. Keep a max-heap of `(−count, char)`. Hold the last used character out for one turn (`prev`), then put it back with its reduced count. If the result is shorter than the input, it is impossible.

In [5]:
def reorganize_string(s):
    heap = [(-c, ch) for ch, c in Counter(s).items()]
    heapq.heapify(heap)
    res, prev = [], (0, "")
    while heap:
        c, ch = heapq.heappop(heap)
        res.append(ch)
        if prev[0] < 0:
            heapq.heappush(heap, prev)           # last char becomes available again
        prev = (c + 1, ch)
    out = "".join(res)
    return out if len(out) == len(s) else ""

print(reorganize_string("aab"), repr(reorganize_string("aaab")))   # aba ''

aba ''


## 6. Interview thinking

**Signals:** "next smallest across several sources", "minimum number of rooms/machines", "always pick the largest/smallest available", "cooldown", "no two adjacent same".

**Say:** "A heap lets me pick the best candidate in O(log n) while candidates change. Time O(n log n) overall."

**Common mistakes:** pushing the whole array instead of one element per list, forgetting to sort meetings by start time, popping instead of replacing for the room reuse, forgetting to hold the previous character out in reorganize string.

## Quick Revision

- Merge k: heap of `(value, list, index)`, push the next from the same list.
- Meeting rooms: sort by start, heap of end times, reuse if `heap[0] <= start`.
- Ropes: always join the two smallest.
- Task scheduler formula: `max(len, (max_freq-1)*(n+1)+count_max)`.
- Reorganize: max-heap + hold previous out one turn.